# Bus Delay DSS - Expanded Self-Contained Notebook

## دفتر Jupyter موسع ومكتمل لرحلة Team B

هذا الدفتر يعيد بناء منطق المشروع داخل ملف واحد حتى يستطيع الطالب قراءة الرحلة كاملة بدون التنقل بين ملفات كثيرة. الهدف ليس استبدال repo، بل فهمه: كيف نبدأ من مشكلة تأخر الحافلات وضعف الثقة، ثم نحولها إلى بيانات، فحص جودة، مؤشرات KPI، سيناريوهات، وتوصية قابلة للمراجعة.

**مهم:** هذا الدفتر لا يستورد `backend.pipeline` أو أي ملف من Backend. الكود منسوخ هنا بشكل تعليمي حتى يرى الطالب كل خطوة في نفس المكان. بعد فهم الدفتر، يعود الطالب إلى repo ليرى أين يعيش نفس المنطق في الملفات الأصلية.

## 1. خريطة الدرس

سنمر على المسار الكامل:

1. تعريف المشكلة والقرار الذي يدعمه النظام.
2. تعريف قواعد توليد Synthetic Data.
3. بناء الجداول الأساسية: `routes`, `stops`, `trips`, `stop_events`.
4. فهم معنى grain والعلاقات بين الجداول.
5. فحص جودة البيانات قبل التحليل.
6. حساب KPIs التي تغذي Dashboard.
7. تشغيل Scenario Lab بطريقة what-if.
8. بناء Recommendation مشروطة بالدليل والثقة والقيود.
9. تصدير مخرجات يمكن استخدامها كدليل.

الفكرة البرمجية الأساسية:

`Config -> Python functions -> DataFrames -> Validation -> Analysis -> Scenarios -> Recommendation -> Evidence`

In [ ]:
from __future__ import annotations

import json
import random
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any

import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)

try:
    display
except NameError:
    def display(value):
        print(value)

## 2. Project context: ما القرار الذي ندعمه؟

A DSS is not just a dashboard. The dashboard shows information, but the DSS must help someone make or review a decision.

في هذا المشروع السؤال هو:

**Which operational intervention should be tested first on Route B12, and why?**

لذلك يجب أن يظهر النظام الدليل: أين التأخير؟ هل البيانات جيدة؟ ما baseline؟ ماذا يحدث في كل scenario؟ ولماذا خرجت recommendation بهذا الشكل؟

In [ ]:
project = {
    "project_name": "Bus Delay DSS",
    "route_id": "B12",
    "route_name": "North Station to Civic Center",
    "problem_statement": (
        "Repeated bus delays reduce reliability and public trust. The system helps decide "
        "which stop, period, and intervention should be reviewed first."
    ),
    "decision_owner": "Public Transport Operations Manager",
    "decision_to_support": (
        "Select the first operational intervention to test on route B12 based on delay evidence, "
        "scenario impact, confidence, and feasibility."
    ),
    "scope": {
        "included": [
            "Synthetic training data for one bus route",
            "Stop-level and trip-level delay analysis",
            "Scenario comparison",
            "Explainable recommendation",
        ],
        "excluded": [
            "Real passenger personal data",
            "Live AVL feed",
            "Procurement decision",
            "City-wide network optimization",
        ],
    },
    "assumptions": [
        "Synthetic data is used for training and must be clearly labeled.",
        "On-time threshold is defined in kpi_config.",
        "Recommendation requires a passing data-quality score.",
    ],
}

decision_card = {
    "decision_id": "DSS-B12-001",
    "decision_title": "First intervention to reduce repeated delay on route B12",
    "decision_question": "Which operational intervention should be tested first, and why?",
    "minimum_quality_score": 0.85,
    "minimum_trips_for_recommendation": 10,
    "evidence_required": [
        "Data quality score",
        "Worst stop ranking",
        "Baseline KPIs",
        "Scenario comparison",
        "Recommendation confidence",
        "Limitations",
    ],
    "safe_failure_message": "Recommendation is blocked until the dataset quality and evidence thresholds pass.",
}

project

## 3. Config: القواعد التي تتحكم في السلوك

هذه القيم موجودة في repo داخل `config/*.json`. وضعها هنا يجعل الدفتر self-contained.

الفكرة المهمة للطلاب: ليس كل تعديل يجب أن يكون داخل Python code. أحيانا نغير assumption أو threshold أو scenario من config، ثم يعيد النظام الحساب.

In [ ]:
generation_rules = {
    "seed": 42,
    "route_id": "B12",
    "trip_count": 36,
    "bus_capacity": 60,
    "service_start_hour": 7,
    "headway_minutes": 15,
    "stops": [
        {"stop_id": "S01", "stop_name": "North Station", "base_boarding": 8, "problem_weight": 0.1},
        {"stop_id": "S02", "stop_name": "Market Street", "base_boarding": 14, "problem_weight": 0.3},
        {"stop_id": "S03", "stop_name": "Central Hospital", "base_boarding": 22, "problem_weight": 0.9},
        {"stop_id": "S04", "stop_name": "University Gate", "base_boarding": 18, "problem_weight": 0.5},
        {"stop_id": "S05", "stop_name": "Civic Center", "base_boarding": 10, "problem_weight": 0.2},
    ],
    "delay_causes": {
        "boarding": {"base_seconds": 35, "variability": 25},
        "traffic": {"base_seconds": 45, "variability": 35},
        "schedule_gap": {"base_seconds": 25, "variability": 20},
    },
    "peak_hours": [7, 8, 16, 17],
    "synthetic_notice": "Training dataset. Not real passenger or live operations data.",
}

kpi_config = {
    "on_time_threshold_sec": 300,
    "dwell_excess_threshold_sec": 45,
    "high_load_ratio": 0.85,
}

validation_rules = {"passing_score": 0.85}

scenarios_config = {
    "scenarios": [
        {"scenario_id": "SC01", "name": "Organized Boarding", "target_stop_id": "S03", "description": "Staff organize boarding at the most delayed stop.", "saving_factor": 0.55, "cost_factor": 1.2, "feasibility": 0.9},
        {"scenario_id": "SC02", "name": "Ticket Machine", "target_stop_id": "S03", "description": "Add ticket machine to reduce boarding delay.", "saving_factor": 0.45, "cost_factor": 1.8, "feasibility": 0.7},
        {"scenario_id": "SC03", "name": "Additional Bus", "target_stop_id": "S03", "description": "Add an extra bus during peak periods.", "saving_factor": 0.65, "cost_factor": 2.6, "feasibility": 0.55},
        {"scenario_id": "SC04", "name": "Schedule Adjustment", "target_stop_id": "S02", "description": "Adjust headway before the busiest segment.", "saving_factor": 0.35, "cost_factor": 1.0, "feasibility": 0.8},
    ]
}

## 4. Generate the dataset

The generated data model has four tables:

- `routes`: route identity and capacity.
- `stops`: ordered stops on the route.
- `trips`: each scheduled bus trip from start to finish.
- `stop_events`: each bus reaching each stop during each trip.

`stop_events` is the richest table because it contains passenger load, dwell time, delay cause, and accumulated delay.

In [ ]:
def generate_dataset(rules: dict[str, Any]) -> dict[str, pd.DataFrame]:
    """Generate a reproducible synthetic bus-route dataset."""
    rng = random.Random(rules["seed"])
    stops_config = rules["stops"]
    route_id = rules["route_id"]
    bus_capacity = rules["bus_capacity"]
    start_time = datetime(2026, 8, 3, rules["service_start_hour"], 0)

    routes = pd.DataFrame([{
        "route_id": route_id,
        "route_name": "North Station to Civic Center",
        "bus_capacity": bus_capacity,
        "synthetic": True,
    }])

    stops = pd.DataFrame([
        {
            "stop_id": stop["stop_id"],
            "route_id": route_id,
            "stop_name": stop["stop_name"],
            "sequence": index + 1,
            "base_boarding": stop["base_boarding"],
        }
        for index, stop in enumerate(stops_config)
    ])

    trips_rows = []
    event_rows = []

    for trip_index in range(rules["trip_count"]):
        departure = start_time + timedelta(minutes=trip_index * rules["headway_minutes"])
        is_peak = departure.hour in rules["peak_hours"]
        trip_id = f"T{trip_index + 1:03d}"
        cumulative_delay = 0
        passenger_load = 0

        for stop_index, stop in enumerate(stops_config):
            cause = choose_delay_cause(rng, stop["problem_weight"], is_peak)
            boarding = max(0, int(rng.gauss(stop["base_boarding"] * (1.35 if is_peak else 1.0), 4)))
            alighting = max(0, int(rng.gauss(4 + stop_index, 2)))
            passenger_load = min(bus_capacity, max(0, passenger_load + boarding - alighting))

            expected_dwell = 25 + boarding * 1.4
            issue_wait = cause_delay_seconds(rng, rules["delay_causes"][cause], stop["problem_weight"], is_peak)
            dwell_time = int(expected_dwell + issue_wait)
            traffic_delay = int(max(0, rng.gauss(35 if is_peak else 15, 12)))
            added_delay = issue_wait + traffic_delay
            cumulative_delay += int(added_delay)

            event_rows.append({
                "trip_id": trip_id,
                "route_id": route_id,
                "stop_id": stop["stop_id"],
                "stop_name": stop["stop_name"],
                "sequence": stop_index + 1,
                "scheduled_arrival_min": stop_index * 6,
                "actual_arrival_min": stop_index * 6 + round(cumulative_delay / 60, 1),
                "passengers_boarding": boarding,
                "passengers_alighting": alighting,
                "passenger_load": passenger_load,
                "bus_capacity": bus_capacity,
                "expected_dwell_sec": round(expected_dwell, 1),
                "dwell_time_sec": dwell_time,
                "dwell_excess_sec": round(max(0, dwell_time - expected_dwell), 1),
                "issue_wait_sec": int(issue_wait),
                "traffic_delay_sec": traffic_delay,
                "added_delay_sec": int(added_delay),
                "cumulative_delay_sec": cumulative_delay,
                "delay_cause": cause,
                "is_peak": is_peak,
            })

        trips_rows.append({
            "trip_id": trip_id,
            "route_id": route_id,
            "departure_time": departure.isoformat(timespec="minutes"),
            "is_peak": is_peak,
            "final_delay_sec": cumulative_delay,
        })

    return {
        "routes": routes,
        "stops": stops,
        "trips": pd.DataFrame(trips_rows),
        "stop_events": pd.DataFrame(event_rows),
    }


def choose_delay_cause(rng: random.Random, problem_weight: float, is_peak: bool) -> str:
    boarding_probability = min(0.75, 0.25 + problem_weight * 0.45 + (0.15 if is_peak else 0))
    roll = rng.random()
    if roll < boarding_probability:
        return "boarding"
    if roll < boarding_probability + 0.18:
        return "traffic"
    return "schedule_gap"


def cause_delay_seconds(rng: random.Random, cause_config: dict[str, float], problem_weight: float, is_peak: bool) -> int:
    base = cause_config["base_seconds"]
    variability = cause_config["variability"]
    peak_factor = 1.35 if is_peak else 1.0
    return max(0, int(rng.gauss(base * problem_weight * peak_factor, variability)))


dataset = generate_dataset(generation_rules)
{name: frame.shape for name, frame in dataset.items()}

## 5. Inspect the tables

قبل أي KPI، افتح الجداول. السؤال التعليمي هنا: ما الذي يمثله الصف الواحد؟

In [ ]:
print("routes")
display(dataset["routes"])

print("stops")
display(dataset["stops"])

print("trips sample")
display(dataset["trips"].head(8))

print("stop_events sample")
display(dataset["stop_events"].head(12))

## 6. Data model and relationships

Relationship:

`routes -> trips -> stop_events <- stops`

- One route has many trips.
- One route has many stops.
- One trip has many stop events.
- One stop appears in many stop events.

In [ ]:
data_model = pd.DataFrame([
    {"table": "routes", "grain": "one bus route", "primary_key": "route_id", "important_fields": "route_id, route_name, bus_capacity"},
    {"table": "stops", "grain": "one stop on the route", "primary_key": "stop_id", "important_fields": "stop_id, route_id, sequence, base_boarding"},
    {"table": "trips", "grain": "one bus trip", "primary_key": "trip_id", "important_fields": "trip_id, route_id, departure_time, final_delay_sec"},
    {"table": "stop_events", "grain": "one trip reaching one stop", "primary_key": "trip_id + stop_id", "important_fields": "trip_id, stop_id, passenger_load, added_delay_sec, delay_cause"},
])

data_model

## 7. Quality gate before analysis

Validation protects the DSS. If the data is broken, the recommendation should be blocked or weakened instead of pretending everything is fine.

In [ ]:
def validate_dataset(dataset: dict[str, pd.DataFrame], rules: dict[str, Any]) -> dict[str, Any]:
    checks = []
    expected_tables = {"routes", "stops", "trips", "stop_events"}
    checks.append(check("required_tables", expected_tables.issubset(dataset.keys()), "All expected tables exist."))

    events = dataset.get("stop_events", pd.DataFrame())
    trips = dataset.get("trips", pd.DataFrame())
    stops = dataset.get("stops", pd.DataFrame())

    checks.append(check("valid_trip_references", events.empty or events["trip_id"].isin(set(trips["trip_id"])).all(), "Every stop event references an existing trip."))
    checks.append(check("valid_stop_references", events.empty or events["stop_id"].isin(set(stops["stop_id"])).all(), "Every stop event references an existing stop."))

    numeric_cols = ["dwell_time_sec", "added_delay_sec", "cumulative_delay_sec", "passenger_load"]
    non_negative = all((events[col] >= 0).all() for col in numeric_cols if col in events)
    checks.append(check("non_negative_time", non_negative, "Time, load, and delay values are not negative."))

    capacity_ok = events.empty or (events["passenger_load"] <= events["bus_capacity"]).all()
    checks.append(check("capacity_not_exceeded", capacity_ok, "Passenger load does not exceed bus capacity.", severity="warning"))

    ordered = True
    if not events.empty:
        for _, group in events.sort_values(["trip_id", "sequence"]).groupby("trip_id"):
            if not group["sequence"].is_monotonic_increasing:
                ordered = False
                break
    checks.append(check("ordered_stop_sequence", ordered, "Stop sequence increases within each trip."))

    passed_count = sum(1 for item in checks if item["passed"])
    score = round(passed_count / len(checks), 2)
    passing_score = rules["passing_score"]

    return {
        "quality_score": score,
        "passing_score": passing_score,
        "passed": score >= passing_score and all(item["passed"] for item in checks if item["severity"] == "critical"),
        "checks": checks,
        "failed_checks": [item for item in checks if not item["passed"]],
    }


def check(check_id: str, passed: bool, message: str, severity: str = "critical") -> dict[str, Any]:
    return {"id": check_id, "passed": bool(passed), "severity": severity, "message": message}


validation_report = validate_dataset(dataset, validation_rules)
display(pd.DataFrame(validation_report["checks"]))
validation_report

## 8. KPI calculation

KPIs are decision signals, not decorations. Each KPI should be traceable to a question, table, fields, formula, and limitation.

In [ ]:
def records(frame: pd.DataFrame) -> list[dict[str, Any]]:
    return frame.round(2).to_dict(orient="records")


def analyze_dataset(dataset: dict[str, pd.DataFrame], kpi_config: dict[str, Any]) -> dict[str, Any]:
    trips = dataset["trips"]
    events = dataset["stop_events"]
    threshold = kpi_config["on_time_threshold_sec"]

    stop_summary = (
        events.groupby(["stop_id", "stop_name"], as_index=False)
        .agg(
            avg_added_delay_sec=("added_delay_sec", "mean"),
            avg_dwell_excess_sec=("dwell_excess_sec", "mean"),
            avg_passenger_load=("passenger_load", "mean"),
            peak_events=("is_peak", "sum"),
        )
        .sort_values("avg_added_delay_sec", ascending=False)
    )

    worst_stop = stop_summary.iloc[0].to_dict()
    cause_summary = (
        events.groupby("delay_cause", as_index=False)
        .agg(total_added_delay_sec=("added_delay_sec", "sum"), events=("trip_id", "count"))
        .sort_values("total_added_delay_sec", ascending=False)
    )

    return {
        "kpis": {
            "trips_analyzed": int(len(trips)),
            "on_time_rate": round(float((trips["final_delay_sec"] <= threshold).mean()), 2),
            "avg_final_delay_sec": round(float(trips["final_delay_sec"].mean()), 1),
            "avg_dwell_time_sec": round(float(events["dwell_time_sec"].mean()), 1),
            "worst_stop": worst_stop["stop_name"],
            "worst_stop_id": worst_stop["stop_id"],
        },
        "stop_summary": records(stop_summary),
        "cause_summary": records(cause_summary),
        "trip_summary": records(trips.sort_values("final_delay_sec", ascending=False).head(10)),
    }


analysis = analyze_dataset(dataset, kpi_config)
analysis["kpis"]

In [ ]:
print("Stop ranking")
display(pd.DataFrame(analysis["stop_summary"]))

print("Delay cause summary")
display(pd.DataFrame(analysis["cause_summary"]))

print("Worst trips")
display(pd.DataFrame(analysis["trip_summary"]))

## 9. Scenario Lab: what-if testing

A scenario is a controlled change to the dataset. It is not a promise. It asks:

**If we apply this intervention, what might improve, and how confident are we?**

The scenario should copy baseline data, apply a limited change, recalculate KPIs, then compare.

In [ ]:
def run_scenarios(dataset: dict[str, pd.DataFrame], scenarios_config: dict[str, Any], kpi_config: dict[str, Any]) -> list[dict[str, Any]]:
    baseline = analyze_dataset(dataset, kpi_config)["kpis"]
    results = []

    for scenario in scenarios_config["scenarios"]:
        adjusted = apply_scenario(dataset["stop_events"], scenario)
        scenario_dataset = {
            **dataset,
            "stop_events": adjusted,
            "trips": recalculate_trips(dataset["trips"], adjusted),
        }
        scenario_kpis = analyze_dataset(scenario_dataset, kpi_config)["kpis"]
        improvement = baseline["avg_final_delay_sec"] - scenario_kpis["avg_final_delay_sec"]
        confidence = confidence_score(improvement, scenario["feasibility"], scenario["cost_factor"])
        action_score = round((max(improvement, 0) * scenario["feasibility"] * confidence) / scenario["cost_factor"], 2)

        results.append({
            "scenario_id": scenario["scenario_id"],
            "name": scenario["name"],
            "description": scenario["description"],
            "target_stop_id": scenario["target_stop_id"],
            "baseline_avg_delay_sec": baseline["avg_final_delay_sec"],
            "scenario_avg_delay_sec": scenario_kpis["avg_final_delay_sec"],
            "improvement_sec": round(improvement, 1),
            "confidence": confidence,
            "cost_factor": scenario["cost_factor"],
            "feasibility": scenario["feasibility"],
            "action_score": action_score,
        })

    return sorted(results, key=lambda item: item["action_score"], reverse=True)


def apply_scenario(events: pd.DataFrame, scenario: dict[str, Any]) -> pd.DataFrame:
    adjusted = events.copy()
    float_columns = ["dwell_time_sec", "dwell_excess_sec", "added_delay_sec", "cumulative_delay_sec"]
    adjusted[float_columns] = adjusted[float_columns].astype(float)

    mask = adjusted["stop_id"] == scenario["target_stop_id"]
    saving = adjusted.loc[mask, "issue_wait_sec"] * scenario["saving_factor"]

    adjusted.loc[mask, "dwell_time_sec"] = (adjusted.loc[mask, "dwell_time_sec"] - saving).clip(lower=0)
    adjusted.loc[mask, "dwell_excess_sec"] = (
        adjusted.loc[mask, "dwell_time_sec"] - adjusted.loc[mask, "expected_dwell_sec"]
    ).clip(lower=0)
    adjusted.loc[mask, "added_delay_sec"] = (adjusted.loc[mask, "added_delay_sec"] - saving).clip(lower=0)
    adjusted["cumulative_delay_sec"] = adjusted.groupby("trip_id")["added_delay_sec"].cumsum()
    return adjusted


def recalculate_trips(trips: pd.DataFrame, events: pd.DataFrame) -> pd.DataFrame:
    final_delay = events.sort_values("sequence").groupby("trip_id").tail(1)[["trip_id", "cumulative_delay_sec"]]
    recalculated = trips.drop(columns=["final_delay_sec"]).merge(final_delay, on="trip_id")
    return recalculated.rename(columns={"cumulative_delay_sec": "final_delay_sec"})


def confidence_score(improvement: float, feasibility: float, cost_factor: float) -> float:
    if improvement <= 0:
        return 0.2
    return round(min(0.95, 0.45 + feasibility * 0.35 + min(improvement / 600, 0.2) - cost_factor * 0.03), 2)


scenario_results = run_scenarios(dataset, scenarios_config, kpi_config)
pd.DataFrame(scenario_results)

## 10. Recommendation logic

Recommendation يجب أن تتضمن الدليل والثقة والقيود. إذا فشلت quality gate، يجب أن يتوقف النظام بأمان.

In [ ]:
def generate_recommendation(
    project: dict[str, Any],
    decision_card: dict[str, Any],
    validation_report: dict[str, Any],
    analysis: dict[str, Any],
    scenario_results: list[dict[str, Any]],
) -> dict[str, Any]:
    if not validation_report["passed"]:
        return {
            "status": "blocked",
            "reason": decision_card["safe_failure_message"],
            "evidence": {
                "quality_score": validation_report["quality_score"],
                "failed_checks": validation_report["failed_checks"],
            },
            "next_step": "Fix data-quality issues before analysis and recommendation.",
        }

    trips_analyzed = analysis["kpis"]["trips_analyzed"]
    if trips_analyzed < decision_card["minimum_trips_for_recommendation"]:
        return {
            "status": "blocked",
            "reason": "Not enough trips to support a recommendation.",
            "evidence": {"trips_analyzed": trips_analyzed},
            "next_step": "Generate more trips or reduce the minimum threshold only with supervisor approval.",
        }

    best = scenario_results[0]
    confidence = best["confidence"]
    status = "recommendation_ready" if confidence >= 0.65 else "needs_review"

    return {
        "status": status,
        "project": project["project_name"],
        "decision_question": decision_card["decision_question"],
        "recommended_action": best["name"],
        "target_stop_id": best["target_stop_id"],
        "expected_improvement_sec": best["improvement_sec"],
        "confidence": confidence,
        "action_score": best["action_score"],
        "evidence": {
            "quality_score": validation_report["quality_score"],
            "worst_stop": analysis["kpis"]["worst_stop"],
            "baseline_avg_delay_sec": best["baseline_avg_delay_sec"],
            "scenario_avg_delay_sec": best["scenario_avg_delay_sec"],
            "scenario_description": best["description"],
        },
        "limitations": [
            "Synthetic training data only.",
            "No live AVL feed.",
            "Cost and feasibility are simplified for learning.",
            "Recommendation should be piloted before operational adoption.",
        ],
        "next_step": "Run a controlled pilot and compare before/after KPIs.",
        "audit_note": "Recommendation generated from config, synthetic data, validation, analysis, and scenario comparison.",
    }


recommendation = generate_recommendation(project, decision_card, validation_report, analysis, scenario_results)
recommendation

## 11. Full pipeline as one function

بعد فهم كل جزء، نعيد جمع الرحلة في دالة واحدة. هذا يعكس طريقة عمل Backend API.

In [ ]:
def run_full_pipeline(write_outputs: bool = False) -> dict[str, Any]:
    dataset = generate_dataset(generation_rules)
    validation_report = validate_dataset(dataset, validation_rules)
    analysis = analyze_dataset(dataset, kpi_config)
    scenario_results = run_scenarios(dataset, scenarios_config, kpi_config)
    recommendation = generate_recommendation(project, decision_card, validation_report, analysis, scenario_results)

    result = {
        "project": project,
        "dataset": dataset,
        "validation": validation_report,
        "analysis": analysis,
        "scenarios": scenario_results,
        "recommendation": recommendation,
    }

    if write_outputs:
        output_dir = Path("notebook_outputs")
        output_dir.mkdir(exist_ok=True)
        for name, frame in dataset.items():
            frame.to_csv(output_dir / f"{name}.csv", index=False)
        (output_dir / "validation_report.json").write_text(json.dumps(validation_report, indent=2, ensure_ascii=False), encoding="utf-8")
        (output_dir / "analysis_summary.json").write_text(json.dumps(analysis, indent=2, ensure_ascii=False), encoding="utf-8")
        (output_dir / "scenario_results.json").write_text(json.dumps(scenario_results, indent=2, ensure_ascii=False), encoding="utf-8")
        (output_dir / "recommendation.json").write_text(json.dumps(recommendation, indent=2, ensure_ascii=False), encoding="utf-8")

    return result


full_result = run_full_pipeline(write_outputs=False)
full_result["recommendation"]

## 12. Final student checkpoints

قبل إنهاء الدفتر، يجب أن يستطيع الطالب شرح:

1. لماذا `stop_events` هو الجدول الأغنى.
2. لماذا validation قبل recommendation.
3. كيف يحسب `on_time_rate`.
4. لماذا worst stop دليل وليس قرار نهائي وحده.
5. ماذا يغير scenario في البيانات.
6. لماذا `confidence`, `feasibility`, و`cost_factor` تؤثر في ranking.
7. لماذا recommendation تحتاج limitations.
8. كيف يعود كل جزء في هذا الدفتر إلى ملف في repo وصفحة في Frontend.

In [ ]:
summary = {
    "project": project["project_name"],
    "route": project["route_id"],
    "tables": {name: len(frame) for name, frame in dataset.items()},
    "quality_score": validation_report["quality_score"],
    "kpis": analysis["kpis"],
    "best_scenario": scenario_results[0],
    "recommendation_status": recommendation["status"],
    "recommended_action": recommendation.get("recommended_action"),
    "next_step": recommendation.get("next_step"),
}

summary

## 13. Map notebook sections back to repo files

| Notebook section | Repo file | Frontend page |
|---|---|---|
| Project context | `config/problem_scope.json` | Decision Brief |
| Data generation | `backend/generator.py` | Synthetic Data |
| Validation | `backend/validation.py` | Quality Gate |
| KPI analysis | `backend/analysis.py` | Dashboard + KPI Builder |
| Scenario comparison | `backend/scenarios.py` | Scenario Lab |
| Recommendation | `backend/recommendation.py` | Decision Output |
| Full pipeline | `backend/pipeline.py` | All pages through API |
| API contract | `backend/main.py` | API learning panels |

The notebook is the learning path. The repo is the product implementation.